# Running Quantum Circuits with Qiskit C++
This tutorial introduces how to run quantum circuits through sampler interface with Qiskit C++.
To run the quantum circuits with Qiskit C++, make sure you need an [IBM Quantum Platform](https://quantum.cloud.ibm.com/) (IQP) account.


## Setting for This Tutorial
In addition to the previous tutorial, the path to `QRMI` is also needed to be set in `#pragma` for `xeus-cling`.

In [1]:
// setting include path to Qiskit C-API, set your own path here
#pragma cling add_include_path("../../qiskit/dist/c/include")
// setting library path to Qiskit C-API, set your own path here
#pragma cling add_library_path("../../qiskit/dist/c/lib")
// loading Qiskit
#pragma cling load("qiskit")

// setting include path to QRMI, set your own path here
#pragma cling add_include_path("../../qrmi")
// setting library path to QRMI, set your own path here
#pragma cling add_library_path("../../qrmi/target/release")
// loading QRMI
#pragma cling load("qrmi")

// then set include path for Qiskit C++
#pragma cling add_include_path("../src")

## Setting Your IQP Account
Qiskit C++ accepts IQP account settings as similar to [Qiskit IBM Runtime](https://github.com/Qiskit/qiskit-ibm-runtime). You will need to set your `API key` and `Instance CRN` with one of the followings.

### Setting in Environment Variables
You can set your account information in environment variables as:
```
export QISKIT_IBM_TOKEN="Your API key"
export QISKIT_IBM_INSTANCE="Your CRN"
```

### Using Saved Account Informations
If you have previously used Qiskit IBM Runtime client, you already have saved account information in `$HOME/.qiskit/qiskit-ibm.json` see [here](https://github.com/Qiskit/qiskit-ibm-runtime?tab=readme-ov-file#save-your-account-on-disk) to save your account in the file.


```
{
    "default-ibm-quantum-platform": {
        "channel": "ibm_quantum_platform",
        "instance": "Your CRN",
        "token": "Your API key",
        "url": "https://cloud.ibm.com"
    }
}
```

### Pass by Parameters
Or you can pass `API key` and `CRN` to the parameters to initialize `QiskitRuntimeService` class of Qiskit C++.
```
#include "service/qiskit_runtime_service qrmi.hpp"
auto service = Qiskit::service::QiskitRuntimeService("Your API key here", "Your CRN here");
```

## Making a GHZ Circuit
In this tutorial, we use a simple circuit to create [GHZ state](https://en.wikipedia.org/wiki/Greenberger%E2%80%93Horne%E2%80%93Zeilinger_state).

In [2]:
#include "circuit/quantumcircuit.hpp"
using namespace Qiskit::circuit;

int num_qubits = 10;
auto qreg = QuantumRegister(num_qubits);
auto creg = ClassicalRegister(num_qubits);
QuantumCircuit circ(std::vector<QuantumRegister>({qreg,}), std::vector<ClassicalRegister>({creg, }));

circ.h(0);
for (int i = 1; i < num_qubits; i++) {
    circ.cx(0, i);
}
circ.measure(qreg, creg);

## Using Runtime Service and Transpiling a Quantum Circuit
To run the quantum circuit on the quantum device, the input circuit should be transpiled for the target backend device, because supported quantum gates are depending on the target device and quantum gates should be mapped to the topology of qubits on the target device.

A `QiskitRuntimeService` class provides interface to the target backend device to transpile and run the quantum circuits. `QiskitRuntimeService` class can be accessed by including one of the runtime interface header file `qiskit_runtime_service_qrmi.hpp` or `qiskit_runtime_service_c.hpp` or `qiskit_runtime_service_sqc.hpp`. (We use `qiskit_runtime_service_qrmi.hpp` for this tutorial)

We can get backend by the name of the backend device, you can find the name of backends you can access on the IQP dashboard.

In [ ]:
#include "service/qiskit_runtime_service qrmi.hpp"

using namespace Qiskit::service;
auto service = QiskitRuntimeService();
// get backend by name, set your backend name here
auto backend = service.backend("ibm_fez");

QRMI connecting : ibm_fez


Then GHZ circuit is transpiled for the target device.

In [ ]:
#include "compiler/transpiler.hpp"
using namespace Qiskit::compiler;
auto transpiled_circ = transpile(circ, backend);

## Running GHZ Circuit with Sampler Interface

Pass the transpiled circuit to `run` function in the `SamplerPub` format to run the quantum circuit.
`run` function immediately return the `Job` class so you can asynchronously run the quantum circuit, so you can do other computation on the classical computer while waiting for the job will be finished.

In [ ]:
#include "primitives/backend_sampler_v2.hpp"
using namespace Qiskit::primitives;
using Sampler = BackendSamplerV2;
// initialize sampler interface with target backend and default number of shots
auto sampler = Sampler(backend, 1000);
// run transpiled circuit
auto job = sampler.run({SamplerPub(transpiled_circ)});
// you can do something here while job will be finished

 QRMI Job submitted to ibm_fez, JOB ID = d4tmfusgk3fc73at2tkg


On the IQP dashboard, submitted quantum circuit can be seen like this.
![Submitted Quantum Circuit on IQP dash board](hgz_on_iqp.png)

## Get the Sampler Result

`result()` function of `Job` class is a blocking function that waits until the job will be finished. It will return the sampler result when the job will be finished. Or you can also check the status of the job by using `status()` function.

In [ ]:
// waiting for the job will be finished and get result
auto result = job->result();

`PrimitiveResult` class returned from `Job` contains sampling results for all pubs (quantum circuits). We can access by using `[]` operator to get `SamplerPubResult` for each pub. 

In [ ]:
// get result for the first pub (only 1 pub in this example)
auto pub_result = result[0];

`SamplerPubResult` class has sampled bit strings. `SamplerPubResult::data` function returns bit strings measured on the classical register in `BitArray` class. `BitArray` contains a list of bit strings and `BitArray::get_bitstrings` function returns `std::vector<std::string>` formatted bit strings. 

In [ ]:
// get the first classical register (only 1 in this example)
auto meas_bits = pub_result.data();
// list of string (binary number in text)
auto bits = meas_bits.get_bitstrings();
std::cout << " ===== samples for pub[0] =====" << std::endl;
for (auto b : bits) {
    std::cout << b << ", ";
}
std::cout << std::endl;

 ===== samples for pub[0] =====
0100000000, 0001111111, 1111111111, 1111111111, 1111111111, 0000000000, 0000000000, 1111111111, 1111111111, 0000000000, 0000000000, 0000000000, 0000000000, 0000000000, 0000000000, 1111111111, 1111111111, 1111111111, 0000000000, 0000000000, 1111111111, 0000000000, 1111111111, 1100000000, 0000000000, 1111111111, 1111111111, 1111111111, 0000000000, 0000000000, 1111111110, 1111111111, 1111111111, 0000000000, 0000000000, 1111111111, 1111111111, 0000000000, 1111111111, 0111111111, 0000001000, 0000000000, 0000000000, 1111111111, 0000001000, 0000001000, 1111111110, 1111011111, 0000000000, 1111111111, 0000000000, 0000000000, 1111111111, 1111111111, 0000000000, 1100000000, 1000000000, 1111111111, 0000000000, 0000000000, 1111111111, 1111111111, 1000000000, 0000000000, 1111111111, 1111111111, 0000000001, 1111111111, 0000000010, 0000000000, 1111111111, 1111111111, 1111111111, 0000000000, 0000000000, 1110111111, 0000000000, 1111111110, 0000000000, 0000000000, 11111111

Also we can get counts for each possibility in a list by using `BitArray::get_counts` function.

In [ ]:
auto counts = meas_bits.get_counts();
std::cout << " ===== counts for pub[0] =====" << std::endl;
for (auto c : counts) {
    // counts is pair of (possibility bitstring, count)
    std::cout << c.first << " : " << c.second << std::endl;
}

 ===== counts for pub[0] =====
1000111111 : 1
0000111111 : 1
0001000000 : 1
0011111111 : 2
0000010000 : 1
0111111011 : 1
1111110000 : 2
1111000000 : 3
0110111111 : 2
1111100000 : 2
1111111011 : 3
0010000001 : 1
1111111110 : 21
0000001000 : 5
0111101111 : 1
1110111111 : 7
1111111100 : 1
0111111111 : 22
1111111000 : 6
0010000000 : 2
1100000000 : 7
0000000000 : 433
0001111111 : 3
0000000010 : 2
0000000001 : 4
1111111111 : 377
1111111101 : 5
0100000000 : 2
1001111111 : 1
0000011111 : 3
1011111111 : 1
1111011111 : 5
0000000111 : 5
0000000110 : 1
1111101111 : 8
0111111110 : 2
0000000011 : 3
1101110111 : 1
0011111101 : 1
0000001111 : 1
0000000100 : 4
1000000000 : 26
1110000000 : 1
1101111111 : 9
1111111010 : 1
1111110111 : 8
1010000000 : 1


***
[back to top](readme.md)